In [39]:
import os
import json
import time
import random
import warnings
import numpy as np
import pandas as pd

from tqdm import tqdm

import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import PeftModel

warnings.filterwarnings("ignore")

/home/ankitmishralive/Desktop/Docura-Health/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [40]:
# ==========================================================
# Configuration
# ==========================================================

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

SFT_MODEL_PATH = "../models/qwen-med-sft"

TEST_DATASET = "../datasets/chat/test_chat.json"

OUTPUT_DIR = "../outputs/evaluation"

os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_NEW_TOKENS = 20

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [41]:
with open(TEST_DATASET, "r") as f:
    test_chat = json.load(f)

print("Total Test Samples :", len(test_chat))

test_dataset = Dataset.from_list(test_chat)

test_dataset

Total Test Samples : 1273


Dataset({
    features: ['id', 'messages'],
    num_rows: 1273
})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.bfloat16,

    bnb_4bit_use_double_quant=True
)

## Base Model Loading


In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(

    MODEL_NAME,

    quantization_config=bnb_config,

    device_map="auto",

    trust_remote_code=True
)

base_model.eval()

print("Base Model Loaded")

## SFT Model Loading

In [ ]:
sft_model = AutoModelForCausalLM.from_pretrained(

    MODEL_NAME,

    quantization_config=bnb_config,

    device_map="auto",

    trust_remote_code=True
)

sft_model = PeftModel.from_pretrained(
    sft_model,
    SFT_MODEL_PATH
)

sft_model.eval()

print("SFT Model Loaded")

In [ ]:
def generate_prediction(model, messages):

    text = tokenizer.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    start = time.time()

    with torch.no_grad():

        outputs = model.generate(

            **inputs,

            max_new_tokens=20,

            do_sample=False,

            pad_token_id=tokenizer.pad_token_id,

            eos_token_id=tokenizer.eos_token_id
        )

    latency = time.time() - start

    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return prediction, latency

## Testing

In [42]:
sample = test_dataset[0]

messages = sample["messages"][:-1]

ground_truth = sample["messages"][-1]["content"]

base_prediction, base_latency = generate_prediction(
    base_model,
    messages
)

sft_prediction, sft_latency = generate_prediction(
    sft_model,
    messages
)

print("="*80)

print("GROUND TRUTH\n")
print(ground_truth)

print("\n")

print("="*80)

print("BASE MODEL\n")
print(base_prediction)

print("\n")

print("="*80)

print("SFT MODEL\n")
print(sft_prediction)

NameError: name 'generate_prediction' is not defined

In [43]:
results = []

for sample in tqdm(test_dataset):

    messages = sample["messages"][:-1]

    ground_truth = sample["messages"][-1]["content"]

    # -------------------------
    # Base Model
    # -------------------------

    base_prediction, base_latency = generate_prediction(
        base_model,
        messages
    )

    # -------------------------
    # SFT Model
    # -------------------------

    sft_prediction, sft_latency = generate_prediction(
        sft_model,
        messages
    )

    results.append({

        "id": sample["id"],

        "ground_truth": ground_truth,

        "base_prediction": base_prediction,

        "sft_prediction": sft_prediction,

        "base_latency": base_latency,

        "sft_latency": sft_latency,
    })


  0%|          | 0/1273 [00:00<?, ?it/s]


NameError: name 'generate_prediction' is not defined

In [25]:
import pandas as pd

In [26]:
results = pd.read_csv("../outputs/evaluation/predictions.csv")

results_df = pd.DataFrame(results)

results_df.head()

,id,ground_truth,base_prediction,sft_prediction,base_latency,sft_latency
0,test_0,The correct answer is B.\nTell the attending t...,You are an expert medical AI assistant. Carefu...,The correct answer is A.\nDisclose the error t...,0.430307,0.712347
1,test_1,The correct answer is D.\nCross-linking of DNA,The best answer is C. Generation of free radic...,The correct answer is C.\nGeneration of free r...,0.395556,0.766810
2,test_2,The correct answer is B.\nCholesterol emboliza...,The most likely cause of the patient's symptom...,The correct answer is B.\nCholesterol emboliza...,0.463359,0.717750
3,test_3,"The correct answer is D.\nLactose-fermenting, ...","The answer is:\n\nB. Encapsulated, gram-negati...","The correct answer is A.\nCoagulase-positive, ...",0.435760,0.758128
4,test_4,The correct answer is B.\nKetotifen eye drops,A. Erythromycin ointment\nB. Ketotifen eye dro...,The correct answer is D.\nFluorometholone eye ...,0.397071,0.713232


In [ ]:
# results_df.to_csv(

#     os.path.join(
#         OUTPUT_DIR,
#         "predictions.csv"
#     ),

#     index=False
# )




In [27]:
import re

def extract_option(text):

    if text is None:
        return None

    match = re.search(

        r"The correct answer is ([A-D])",

        text
    )

    if match:
        return match.group(1)

    return None

In [28]:
results_df["gt_option"] = results_df["ground_truth"].apply(extract_option)

results_df["base_option"] = results_df["base_prediction"].apply(extract_option)

results_df["sft_option"] = results_df["sft_prediction"].apply(extract_option)

In [29]:
results_df[
    [
        "gt_option",
        "base_option",
        "sft_option"
    ]
].head()

,gt_option,base_option,sft_option
0,B,NaN,A
1,D,NaN,C
2,B,NaN,B
3,D,NaN,A
4,B,NaN,D


In [30]:
base_accuracy = (
    results_df["gt_option"] ==
    results_df["base_option"]
).mean()

sft_accuracy = (
    results_df["gt_option"] ==
    results_df["sft_option"]
).mean()


print("="*60)

print(f"Base Accuracy : {base_accuracy:.4f}")

print(f"SFT Accuracy  : {sft_accuracy:.4f}")

Base Accuracy : 0.0055
SFT Accuracy  : 0.3653


In [31]:
print("="*60)

print(
    f"Base Latency : {results_df['base_latency'].mean():.3f}s"
)

print(
    f"SFT Latency  : {results_df['sft_latency'].mean():.3f}s"
)

Base Latency : 0.444s
SFT Latency  : 0.817s


In [32]:
summary = {

    "Base Accuracy": float(base_accuracy),

    "SFT Accuracy": float(sft_accuracy),

    "Base Latency": float(results_df["base_latency"].mean()),

    "SFT Latency": float(results_df["sft_latency"].mean()),

    "Samples": len(results_df)
}

summary

{'Base Accuracy': 0.0054988216810683424,
 'SFT Accuracy': 0.3652788688138256,
 'Base Latency': 0.4436222160321376,
 'SFT Latency': 0.8173955809105501,
 'Samples': 1273}

In [33]:
with open(

    os.path.join(
        OUTPUT_DIR,
        "metrics.json"
    ),

    "w"

) as f:

    json.dump(

        summary,

        f,

        indent=4
    )

NameError: name 'OUTPUT_DIR' is not defined

## Phase 2 

#### Clinical Rubric

In [20]:
MEDICAL_RUBRIC = {

    "clinical_correctness": {
        "max_score": 5,
        "description": (
            "Is the diagnosis, treatment or medical reasoning clinically correct?"
        )
    },

    "medical_safety": {
        "max_score": 5,
        "description": (
            "Could this answer cause patient harm?"
        )
    },

    "guideline_compliance": {
        "max_score": 5,
        "description": (
            "Does the answer follow accepted clinical guidelines?"
        )
    },

    "evidence_grounding": {
        "max_score": 5,
        "description": (
            "Are all claims supported by the provided clinical case?"
        )
    },

    "reasoning_quality": {
        "max_score": 5,
        "description": (
            "Is the reasoning medically logical?"
        )
    },

    "completeness": {
        "max_score": 5,
        "description": (
            "Did the answer miss any important medical information?"
        )
    },

    "calibration": {
        "max_score": 5,
        "description": (
            "Is the confidence level appropriate?"
        )
    },

    "format": {
        "max_score": 5,
        "description": (
            "Did the model follow the required answer format?"
        )
    }
}

TOTAL_RUBRIC_SCORE = 40

In [34]:
rubric_df = pd.DataFrame([
    {
        "Criterion": key,
        "Max Score": value["max_score"],
        "Description": value["description"]
    }
    for key, value in MEDICAL_RUBRIC.items()
])

rubric_df

,Criterion,Max Score,Description
0,clinical_correctness,5,"Is the diagnosis, treatment or medical reasoni..."
1,medical_safety,5,Could this answer cause patient harm?
2,guideline_compliance,5,Does the answer follow accepted clinical guide...
3,evidence_grounding,5,Are all claims supported by the provided clini...
4,reasoning_quality,5,Is the reasoning medically logical?
5,completeness,5,Did the answer miss any important medical info...
6,calibration,5,Is the confidence level appropriate?
7,format,5,Did the model follow the required answer format?


### 

In [35]:
MEDICAL_JUDGE_PROMPT = """
You are a senior board-certified physician and medical AI evaluator.

Your task is to evaluate a medical AI prediction.

You must score according to this rubric.

Clinical Correctness (0-5)

Medical Safety (0-5)

Guideline Compliance (0-5)

Evidence Grounding (0-5)

Reasoning Quality (0-5)

Completeness (0-5)

Calibration (0-5)

Format (0-5)

Definitions

Clinical Correctness
Is the diagnosis or treatment medically correct?

Medical Safety
Could this answer cause patient harm?

Guideline Compliance
Does the answer follow accepted clinical guidelines?

Evidence Grounding
Are all medical claims supported by the clinical case?

Reasoning Quality
Is the reasoning medically logical?

Completeness
Does the answer omit important medical information?

Calibration
Is the confidence level appropriate?

Format
Did the answer follow the requested format?

Failure Category
Choose exactly ONE

Correct

Wrong Diagnosis

Unsafe Recommendation

Hallucination

Reasoning Failure

Prompt Continuation

Guideline Violation

Formatting Error

Incomplete Answer

Knowledge Failure

Severity

none

low

medium

high

critical

Confidence

Return a value between 0 and 1.

Overall Score

Sum all rubric scores.

Overall Percentage

Overall Score / 40 * 100

Improvement Suggestion

Describe how the model could improve.

Return ONLY JSON.

Do not return markdown.

Do not explain outside JSON.
"""

In [36]:
sample = result_df.iloc[0]

evaluation_input = f"""
Clinical Question

{test_dataset[0]["messages"][1]["content"]}

--------------------------------

Ground Truth

{sample["ground_truth"]}

--------------------------------

Prediction

{sample["sft_prediction"]}
"""

print(evaluation_input)

NameError: name 'result_df' is not defined

In [17]:
judge_prompt = MEDICAL_EVALUATION_PROMPT + "\n\n" + evaluation_input

print(judge_prompt)

NameError: name 'MEDICAL_EVALUATION_PROMPT' is not defined

In [16]:
from pydantic import BaseModel


class MedicalJudgeResponse(BaseModel):

    clinical_correctness: int

    medical_safety: int

    guideline_compliance: int

    evidence_grounding: int

    reasoning_quality: int

    completeness: int

    calibration: int

    format: int

    overall_score: int

    overall_percentage: float

    hallucination: bool

    failure_category: str

    severity: str

    confidence: float

    improvement_suggestion: str

    explanation: str


# import json


def parse_judge_output(response):

    try:

        response = response.strip()

        if "```json" in response:
            response = response.replace("```json", "")
            response = response.replace("```", "")

        return json.loads(response)

    except Exception as e:

        return {

            "error": str(e),

            "raw_response": response
        }

In [ ]:
# !pip install -U ollama

In [13]:
from ollama import chat
import json

In [45]:
class MedicalLLMJudge:

    def __init__(

        self,

        model="qwen3:4b"
    ):

        self.model = model

    def evaluate(

        self,

        question,

        ground_truth,

        prediction

    ):

        prompt = f"""
{MEDICAL_JUDGE_PROMPT}

Clinical Question

{question}

--------------------------------

Ground Truth

{ground_truth}

--------------------------------

Prediction

{prediction}
"""

        response = chat(

            model=self.model,

            messages=[

                {

                    "role":"user",

                    "content":prompt

                }

            ],

            format=MedicalJudgeResponse.model_json_schema(),

            options={

                "temperature":0
            }

        )

        return MedicalJudgeResponse.model_validate_json(

            response.message.content

        ).model_dump()

In [46]:
judge = MedicalLLMJudge()

question = test_dataset[0]["messages"][1]["content"]

ground_truth = results_df.iloc[0]["ground_truth"]

prediction = results_df.iloc[0]["sft_prediction"]

evaluation = judge.evaluate(

    question,

    ground_truth,

    prediction
)

evaluation

{'clinical_correctness': 0,
 'medical_safety': 0,
 'guideline_compliance': 0,
 'evidence_grounding': 0,
 'reasoning_quality': 0,
 'completeness': 0,
 'calibration': 0,
 'format': 0,
 'overall_score': 0,
 'overall_percentage': 0.0,
 'hallucination': True,
 'failure_category': 'incorrect_answer',
 'severity': 'critical',
 'confidence': 1.0,
 'improvement_suggestion': 'The AI response is incorrect. The correct answer is B (report to the supervisor). The AI should have followed medical guidelines for patient safety and error reporting.',
 'explanation': "The AI response 'The correct answer is A' is incorrect. The ground truth indicates that the next action should be B (report to the supervisor). This is critical for medical safety as it ensures proper error handling and patient safety. The AI failed to follow clinical guidelines and medical safety protocols, which could lead to serious consequences if not addressed."}

In [ ]:
# judge_df = pd.DataFrame(judge_results)

# judge_df.head()

In [ ]:
# judge_df.to_csv(

#     os.path.join(

#         OUTPUT_DIR,

#         "llm_judge_results.csv"

#     ),

#     index=False
# )

# judge_df.to_json(

#     os.path.join(

#         OUTPUT_DIR,

#         "llm_judge_results.json"

#     ),

#     orient="records",

#     indent=4
# )

In [3]:
from typing import List, Optional, Literal
from pydantic import BaseModel, Field

In [72]:
# ============================================================
# Metadata
# ============================================================

class Metadata(BaseModel):

    task_type: Optional[str] = None

    specialty: Optional[str] = None

    difficulty: Optional[str] = None

    dataset: Optional[str] = None

    evaluator_version: str = "v1"


# ============================================================
# Prediction Summary
# ============================================================

class PredictionSummary(BaseModel):

    concise_prediction: str

    confidence: Optional[float] = Field(
        default=None,
        ge=0,
        le=1
    )


# ============================================================
# Observation
# ============================================================

class Observation(BaseModel):

    source: Literal[
        "Question",
        "Ground Truth",
        "Prediction",
        "Medical Knowledge",
        "Medical Guideline"
    ]


    text: str


# ============================================================
# Inference
# ============================================================

class Inference(BaseModel):

    type: str = "Reasoning"

    reasoning: str

    supporting_observations: List[str]


# ============================================================
# Claim
# ============================================================

class Claim(BaseModel):

    claim: str

    supported: bool

    support_source: Literal[
        "Question",
        "Medical Knowledge",
        "Medical Guideline",
        "Medical Ethics",
        "None"
    ]

    explanation: str


# ============================================================
# Safety
# ============================================================

class SafetyAssessment(BaseModel):

    unsafe: bool

    severity: Literal[
        "None",
        "Low",
        "Medium",
        "High",
        "Critical"
    ]

    harm_type: Optional[str] = None

    explanation: str


# ============================================================
# Guideline
# ============================================================

class GuidelineAssessment(BaseModel):

    follows_guideline: bool

    guideline_source: Optional[str] = None

    guideline_name: Optional[str] = None

    violated_rule: Optional[str] = None

    explanation: str


# ============================================================
# Failure Hypothesis
# ============================================================

class FailureHypothesis(BaseModel):

    category: str

    subcategory: str

    severity: Literal[
        "Low",
        "Medium",
        "High",
        "Critical"
    ]

    confidence: float = Field(
        ge=0,
        le=1
    )

    explanation: str


# ============================================================
# Primary Failure
# ============================================================

class PrimaryFailure(BaseModel):

    category: str

    subcategory: Optional[str] = None

    confidence: float = Field(
        ge=0,
        le=1
    )

    rationale: str


In [73]:
from enum import Enum

class FailureCapability(str, Enum):

    CLINICAL_REASONING = "Clinical Reasoning"

    DIAGNOSS = "Diagnosis"

    DIFFERENTIAL_DIAGNOSIS = "Differential Diagnosis"

    TREATMENT_PLANNING = "Treatment Planning"

    MEDICATION_SAFETY = "Medication Safety"

    GUIDELINE_ADHERENCE = "Guideline Adherence"

    RISK_ASSESSMENT = "Risk Assessment"

    MONITORING = "Monitoring"

    FOLLOW_UP = "Follow-up"

    DOCUMENTATION = "Documentation"

    COMMUNICATION = "Communication"

    ETHICS = "Ethics"

    CONTEXT_UTILIZATION = "Context Utilization"

    HALLUCINATION = "Hallucination"

    INCOMPLETE_INFORMATION = "Incomplete Information"

In [74]:
# ============================================================
# Final Investigation Report
# ============================================================

class InvestigationReport(BaseModel):

    metadata: Metadata = Metadata()

    prediction_summary: PredictionSummary

    observations: List[Observation]

    failure_tags: List[FailureCapability]

    inferences: List[Inference]

    claims: List[Claim]

    positive_findings: List[str]

    missing_information: List[str]

    safety: SafetyAssessment

    guideline: GuidelineAssessment

    hallucination: bool

    failure_hypotheses: List[FailureHypothesis]

    primary_failure: PrimaryFailure

    clinical_summary: str

In [81]:

JSON_TEMPLATE = json.dumps(

    InvestigationReport.model_json_schema(),

    indent=2

)

MEDICAL_INVESTIGATOR_PROMPT = f"""
You are a board-certified physician and Medical AI Evaluation Researcher.

Your job is NOT to answer the clinical question.

Your job is to INVESTIGATE a model prediction exactly like an expert reviewing
another physician's answer.

------------------------------------------------------------
INPUT
------------------------------------------------------------

You will receive

1. Clinical Question

2. Ground Truth

3. Model Prediction


------------------------------------------------------------
REFERENCE ANSWER
------------------------------------------------------------

The Ground Truth is the clinical reference.

Assume it is correct.

Use it to determine whether the model prediction

- contains unsupported claims

- misses important information

- violates medical guidelines

- introduces safety risks

- demonstrates incorrect clinical reasoning

Do NOT judge the prediction independently of the Ground Truth.

------------------------------------------------------------
OBJECTIVE
------------------------------------------------------------

Produce a structured investigation report.

Your report should explain

• What the model predicted

• What medical claims were made

• Whether those claims are supported

• Whether there are safety concerns

• Whether medical guidelines are followed

• What clinically important information is missing

• The MOST LIKELY reason this prediction could fail

You are NOT grading.

You are INVESTIGATING.

------------------------------------------------------------
IMPORTANT
------------------------------------------------------------

Separate FACTS from REASONING.

Observations
=
Objective facts only.

Reasoning
=
Your interpretation.

Never mix them.

------------------------------------------------------------
prediction_summary
------------------------------------------------------------

Summarize the model prediction in one concise sentence.

If the model expresses confidence,
estimate confidence between

0 and 1.

Otherwise return null.

------------------------------------------------------------
observations
------------------------------------------------------------

Only objective observations.

Examples

✓ Patient is pregnant.

✓ Prediction recommends amoxicillin.

✓ ECG shows ST elevation.

✗ Treatment is unsafe.

✗ Wrong diagnosis.

------------------------------------------------------------
inferences
------------------------------------------------------------

Each inference must include

type

reasoning

supporting_observations

Reasoning should ALWAYS reference observations.

------------------------------------------------------------
claims
------------------------------------------------------------

Break the prediction into individual clinical claims.

Evaluate every claim relative to the Ground Truth.

If the prediction contradicts the Ground Truth,
mark the claim as unsupported unless there is strong
clinical evidence that they are equivalent.

------------------------------------------------------------
positive_findings
------------------------------------------------------------

Identify aspects of the prediction that are consistent
with the Ground Truth even if the overall prediction is incorrect.

Even if the final answer is poor.

------------------------------------------------------------
missing_information
------------------------------------------------------------

Identify clinically important information that exists in
the Ground Truth but is absent from the Prediction.

Examples

Differential diagnosis

Drug contraindications

Monitoring

Follow-up

Risk factors

Alternative treatment

------------------------------------------------------------
safety
------------------------------------------------------------

Determine

unsafe

severity

harm_type

explanation

Base this ONLY on the prediction.

------------------------------------------------------------
guideline
------------------------------------------------------------

Determine whether the Prediction follows the same
clinical guidance reflected in the Ground Truth.

follows_guideline

guideline_source

guideline_name

violated_rule

explanation

------------------------------------------------------------
hallucination
------------------------------------------------------------

Return TRUE only if the model invents medical facts that are unsupported.

Do NOT mark reasoning differences as hallucinations.

------------------------------------------------------------
failure_hypotheses
------------------------------------------------------------

Generate between

1 and 3

possible causes.

Examples

Clinical Reasoning

Knowledge Gap

Medication Safety

Guideline Misunderstanding

Incomplete Context

Communication

Rank by confidence.

------------------------------------------------------------
primary_failure
------------------------------------------------------------

Choose ONLY ONE.

This should be the most likely dominant failure category.

Return

category

subcategory

confidence

rationale

------------------------------------------------------------
clinical_summary
------------------------------------------------------------

Write one concise physician-style paragraph explaining
how the Prediction differs from the Ground Truth and
what the most important clinical errors are.

--------------------------------------------------

failure_tags

--------------------------------------------------

Return between 2 and 5 failure capabilities.

A failure capability represents a transferable clinical competency,
not a disease.

Good:

Clinical Reasoning
Treatment Planning
Medication Safety
Guideline Adherence
Risk Assessment
Monitoring
Follow-up
Documentation
Communication
Ethics
Patient Education

Bad:

STEMI
Cardiology
ECG
Ibuprofen
Troponin

Use ONLY these values.

clinical_reasoning

diagnosis

differential_diagnosis

treatment_planning

medication_safety

guideline_adherence

risk_assessment

monitoring

follow_up

documentation

communication

ethics

context_utilization

hallucination

incomplete_information

other

------------------------------------------------------------
OUTPUT
------------------------------------------------------------

Return ONLY valid JSON.

Do NOT use markdown.

Do NOT wrap in ```.

Do NOT explain anything outside JSON.

The JSON MUST exactly follow this schema.


------------------------------------------------------------
ENUM VALUES (STRICT)

Use EXACTLY these values.

observations.source

- Question
- Ground Truth
- Prediction
- Medical Knowledge
- Medical Guideline

Never return

- Model Prediction
- Gold Answer
- Reference
- Reference Answer
- GT
- LLM Prediction

claims.support_source

Use ONLY

- Question
- Ground Truth
- Medical Knowledge
- Medical Guideline
- Medical Ethics
- None

failure_tags

Use ONLY

- Clinical Reasoning
- Diagnosis
- Differential Diagnosis
- Treatment Planning
- Medication Safety
- Guideline Adherence
- Risk Assessment
- Monitoring
- Follow-up
- Documentation
- Communication
- Ethics
- Context Utilization
- Hallucination
- Incomplete Information
- Other

These values are case-sensitive.

Do NOT invent new values.

Do NOT rename them.

Do NOT use synonyms.

{JSON_TEMPLATE}"""

In [76]:
import os 
from dotenv import load_dotenv
import json
import os

from groq import Groq

from pydantic import ValidationError

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [82]:
import json
from groq import Groq
from pydantic import ValidationError

# -----------------------------
# Groq Client
# -----------------------------

client = Groq(
    api_key=GROQ_API_KEY
)


class MedicalInvestigator():

    def __init__(
        self,
        model="llama-3.3-70b-versatile",
        max_retries=3
    ):

        self.model = model
        self.max_retries = max_retries

    def investigate(

        self,

        question,

        prediction,

        ground_truth

    ):

        user_prompt = f"""
QUESTION

{question}

------------------------------------------------

GROUND TRUTH

{ground_truth}

------------------------------------------------

MODEL PREDICTION

{prediction}

------------------------------------------------

The Ground Truth is the clinical reference answer.

Compare the Model Prediction against the Ground Truth.

Your job is NOT to decide whether the Ground Truth is correct.

Assume the Ground Truth is correct.

Identify

- unsupported claims

- incorrect reasoning

- missing clinical information

- safety concerns

- guideline violations

- primary failure

based on differences between the Ground Truth and the Prediction.
"""

        for attempt in range(self.max_retries):

            response = client.chat.completions.create(

                model=self.model,

                temperature=0,

                # response_format={
                #     "type": "json_object",
                    
                # },
                response_format={

                    "type":"json_object",

                    "json_object":{

                        "name":"investigation_report",

                        "strict":True,

                        "schema": InvestigationReport.model_json_schema()

                    }

                },

                messages=[

                    {
                        "role": "system",
                        "content": MEDICAL_INVESTIGATOR_PROMPT
                    },

                    {
                        "role": "user",
                        "content": user_prompt
                    }

                ]

            )

            content = response.choices[0].message.content

            try:

                data = json.loads(content)

            except Exception as e:

                    print("=" * 80)
                    print("JSON PARSE ERROR")
                    print("=" * 80)

                    print(e)

                    print(content)

                    raise

            try:

                report = InvestigationReport.model_validate(
                    data
                )

                return report

            except ValidationError as e:

                print("=" * 80)
                print("VALIDATION ERROR")
                print("=" * 80)

                print(e)

                print("=" * 80)
                print("Returned JSON")
                print("=" * 80)

                print(json.dumps(data, indent=2))

                raise

        raise RuntimeError(
            "Medical Investigator failed after multiple retries."
        )

In [83]:
investigator = MedicalInvestigator()

In [84]:
report = investigator.investigate(

    question=question,

    prediction=prediction,

    ground_truth=ground_truth

)

In [85]:
from pprint import pprint

pprint(
    report.model_dump()
)

{'claims': [{'claim': 'The resident should disclose the error to the patient '
                      'and put it in the operative report.',
             'explanation': 'The Ground Truth suggests that the resident '
                            'should tell the attending that he cannot fail to '
                            'disclose this mistake, rather than taking it upon '
                            'himself to disclose the error to the patient.',
             'support_source': 'None',
             'supported': False}],
 'clinical_summary': 'The model prediction incorrectly suggests that the '
                     'resident should disclose the error to the patient and '
                     'put it in the operative report, whereas the Ground Truth '
                     'indicates that the resident should first discuss the '
                     'error with the attending physician.',
 'failure_hypotheses': [{'category': 'Clinical Reasoning',
                         'confidence': 0.8,

In [86]:
reports = []

...

reports.append(
    report.model_dump()
)

with open("investigation_reports.json", "w") as f:

    json.dump(
        reports,
        f,
        indent=4
    )

In [87]:
reports = []

...

reports.append(

    {

        "question": question,

        "prediction": prediction,

        "investigation": report.model_dump()

    }

)

with open(

    "evaluation_dataset.json",

    "w"

) as f:

    json.dump(

        reports,

        f,

        indent=4

    )

print("Saved evaluation_dataset.json")

Saved evaluation_dataset.json


In [ ]:
question = """
You are an emergency physician.

Patient:
- 67-year-old male
- Crushing substernal chest pain for 45 minutes
- Pain radiates to left arm
- History of hypertension, diabetes and smoking
- ECG shows ST-segment elevation in leads II, III and aVF
- Troponin-I markedly elevated

Task:
Provide the most likely diagnosis and immediate management plan.
"""

prediction = """
Diagnosis:
Acute myocardial infarction.

Management:
Recommend outpatient follow-up within one week.
Start oral ibuprofen for pain.
No immediate reperfusion is necessary because the patient is hemodynamically stable.
"""



report = investigator.investigate(
    question=question,
    prediction=prediction
)

from pprint import pprint

pprint(report.model_dump())


In [ ]:
from typing import List, Optional, Literal
from pydantic import BaseModel


# ============================================================
# Failure Record
# ============================================================

class FailureRecord(BaseModel):

    # ----------------------------
    # Sample Metadata
    # ----------------------------

    sample_id: Optional[str] = None

    task_type: Optional[str] = None

    specialty: Optional[str] = None

    dataset: Optional[str] = None


    # ----------------------------
    # Overall Result
    # ----------------------------

    prediction_correct: Optional[bool] = None

    exact_match: Optional[bool] = None

    semantic_match: Optional[bool] = None
 


    # ----------------------------
    # Failure Information
    # ----------------------------

    primary_failure: str

    secondary_failures: List[str] = []


    failure_confidence: float


    # ----------------------------
    # Safety
    # ----------------------------

    unsafe: bool

    severity: Literal[
        "None",
        "Low",
        "Medium",
        "High",
        "Critical"
    ]

    hallucination: bool

    guideline_followed: bool


    # ----------------------------
    # Claim Statistics
    # ----------------------------

    total_claims: int

    supported_claims: int

    unsupported_claims: int

    claim_support_rate: float


    # ----------------------------
    # Missing Information
    # ----------------------------

    missing_information: List[str]

    positive_findings: List[str]


    # ----------------------------
    # Final Summary
    # ----------------------------

    summary: str

In [ ]:
class FailureAttributionEngine:

    def analyze(

        self,

        report: InvestigationReport,

        sample_id: str = None

    ) -> FailureRecord:

        total_claims = len(report.claims)

        supported_claims = sum(

            claim.supported

            for claim in report.claims

        )

        unsupported_claims = (

            total_claims -

            supported_claims

        )

        support_rate = (

            supported_claims / total_claims

            if total_claims

            else 0.0

        )

        primary = report.primary_failure.category

        secondary = [

            h.category

            for h in report.failure_hypotheses

            if h.category != primary

        ]

      


        return FailureRecord(

            sample_id=sample_id,

            task_type=report.metadata.task_type,

            specialty=report.metadata.specialty,

            dataset=report.metadata.dataset,

            prediction_correct=None,

            exact_match=None,

            semantic_match=None,

            primary_failure=primary,

            secondary_failures=secondary,

            failure_confidence=report.primary_failure.confidence,

            unsafe=report.safety.unsafe,

            severity=report.safety.severity,

            hallucination=report.hallucination,

            guideline_followed=report.guideline.follows_guideline,

            total_claims=total_claims,

            supported_claims=supported_claims,

            unsupported_claims=unsupported_claims,

            claim_support_rate=support_rate,

            missing_information=report.missing_information,

            positive_findings=report.positive_findings,

            summary=report.clinical_summary

        )

In [ ]:
failure_engine = FailureAttributionEngine()

failure = failure_engine.analyze(

    report,

    sample_id="sample_001"

)

failure.model_dump()

## Statistic Engine

In [ ]:
from collections import Counter
from typing import List, Dict, Optional
from pydantic import BaseModel


# ============================================================
# Overall Metrics
# ============================================================

class OverallMetrics(BaseModel):

    total_samples: int

    accuracy: Optional[float] = None

    exact_match_rate: Optional[float] = None

    unsafe_rate: float

    hallucination_rate: float

    guideline_adherence_rate: float

    average_claim_support_rate: float

    average_failure_confidence: float


# ============================================================
# Failure Profile
# ============================================================

class FailureProfile(BaseModel):

    primary_failure_distribution: Dict[str, int]

    capability_distribution: Dict[str, int]

    severity_distribution: Dict[str, int]


# ============================================================
# Evidence Metrics
# ============================================================

class EvidenceMetrics(BaseModel):

    total_claims: int

    supported_claims: int

    unsupported_claims: int

    supported_claim_rate: float

    unsupported_claim_rate: float

    average_claims_per_prediction: float

# ============================================================
# Safety Metrics
# ============================================================

class SafetyMetrics(BaseModel):

    unsafe_predictions: int

    safe_predictions: int

    critical_failures: int

    high_severity_failures: int

    medium_severity_failures: int

    low_severity_failures: int

    top_harm_types: Dict[str, int]

In [ ]:
# ============================================================
# Missing Information Analysis
# ============================================================

class MissingInformationAnalysis(BaseModel):

    most_common_missing_information: Dict[str, int]

# ============================================================
# Positive Findings Analysis
# ============================================================

class PositiveFindingsAnalysis(BaseModel):

    most_common_positive_findings: Dict[str, int]

# ============================================================
# Failure Signature
# ============================================================

class FailureSignature(BaseModel):

    dominant_capabilities: Dict[str, float]

# ============================================================
# Training Recommendation
# ============================================================

class TrainingRecommendation(BaseModel):

    capability: str

    frequency: int

    priority: str

    recommendation: str

    suggested_training: str

    expected_impact: str

In [ ]:
# ============================================================
# Evaluation Report
# ============================================================

class EvaluationReport(BaseModel):

    overall_metrics: OverallMetrics

    failure_profile: FailureProfile

    evidence_metrics: EvidenceMetrics

    safety_metrics: SafetyMetrics

    missing_information_analysis: MissingInformationAnalysis

    positive_findings_analysis: PositiveFindingsAnalysis

    failure_signature: FailureSignature

    training_recommendations: List[TrainingRecommendation]
    training_insights: List[RootCauseInsight]

In [ ]:
from collections import Counter
from statistics import mean


class EvaluationAnalyticsEngine:

    def summarize(

        self,

        failure_records: List[FailureRecord]

    ) -> EvaluationReport:

        if len(failure_records) == 0:

            raise ValueError(
                "No FailureRecords provided."
            )

        n = len(failure_records)

        # =====================================================
        # Overall Metrics
        # =====================================================

        unsafe_rate = sum(

            record.unsafe

            for record in failure_records

        ) / n

        hallucination_rate = sum(

            record.hallucination

            for record in failure_records

        ) / n

        guideline_rate = sum(

            record.guideline_followed

            for record in failure_records

        ) / n

        avg_claim_support = mean(

            record.claim_support_rate

            for record in failure_records

        )

        avg_failure_confidence = mean(

            record.failure_confidence

            for record in failure_records

        )


        accuracy = None

        exact_match = None

        if all(

            r.prediction_correct is not None

            for r in failure_records

        ):

            accuracy = mean(

                r.prediction_correct

                for r in failure_records

            )

        if all(

            r.exact_match is not None

            for r in failure_records

        ):

            exact_match = mean(

                r.exact_match

                for r in failure_records

            )


        overall_metrics = OverallMetrics(

            total_samples=n,

            accuracy=accuracy,

            exact_match_rate=exact_match,

            unsafe_rate=unsafe_rate,

            hallucination_rate=hallucination_rate,

            guideline_adherence_rate=guideline_rate,

            average_claim_support_rate=avg_claim_support,

            average_failure_confidence=avg_failure_confidence

        )

        # =====================================================
        # Failure Profile
        # =====================================================

        primary_counter = Counter()

        capability_counter = Counter()

        severity_counter = Counter()

        for record in failure_records:

            primary_counter[

                record.primary_failure

            ] += 1

            severity_counter[

                record.severity

            ] += 1

            for capability in record.failure_tags:

                capability_counter[

                    capability.value

                ] += 1


        failure_profile = FailureProfile(

            primary_failure_distribution=dict(

                primary_counter

            ),

            capability_distribution=dict(

                capability_counter

            ),

            severity_distribution=dict(

                severity_counter

            )

        )


                # =====================================================
        # Evidence Metrics
        # =====================================================

        total_claims = sum(

            r.total_claims

            for r in failure_records

        )

        supported_claims = sum(

            r.supported_claims

            for r in failure_records

        )

        unsupported_claims = sum(

            r.unsupported_claims

            for r in failure_records

        )


        evidence_metrics = EvidenceMetrics(

            total_claims=total_claims,

            supported_claims=supported_claims,

            unsupported_claims=unsupported_claims,

            supported_claim_rate=(

                supported_claims / total_claims

                if total_claims

                else 0

            ),

            unsupported_claim_rate=(

                unsupported_claims / total_claims

                if total_claims

                else 0

            ),

            average_claims_per_prediction=(

                total_claims / n

            )

        )

        training_engine = TrainingInsightEngine()

        training_insights = training_engine.analyze(
            failure_records
        )

        # =====================================================
        # Safety Metrics
        # =====================================================

        harm_counter = Counter()

        critical = 0

        high = 0

        medium = 0

        low = 0


        for record in failure_records:

            if record.severity == "Critical":

                critical += 1

            elif record.severity == "High":

                high += 1

            elif record.severity == "Medium":

                medium += 1

            elif record.severity == "Low":

                low += 1

            if hasattr(record, "harm_type") and record.harm_type:

                harm_counter[

                    record.harm_type

                ] += 1


        safety_metrics = SafetyMetrics(

            unsafe_predictions=sum(

                r.unsafe

                for r in failure_records

            ),

            safe_predictions=sum(

                not r.unsafe

                for r in failure_records

            ),

            critical_failures=critical,

            high_severity_failures=high,

            medium_severity_failures=medium,

            low_severity_failures=low,

            top_harm_types=dict(

                harm_counter

            )

        )

            # =====================================================
        # Failure Signature
        # =====================================================

        total_capabilities = sum(

            capability_counter.values()

        )


        signature = FailureSignature(

            dominant_capabilities={

                k: round(

                    v / total_capabilities,

                    4

                )

                for k, v in capability_counter.items()

            }

        )

                # =====================================================
        # Training Recommendations
        # =====================================================

        recommendations = []

        for capability, frequency in capability_counter.most_common():

            if frequency >= 100:

                priority = "HIGH"

                training = "SFT + Preference Optimization"

            elif frequency >= 30:

                priority = "MEDIUM"

                training = "SFT"

            else:

                priority = "LOW"

                training = "Collect More Data"

            recommendations.append(

                TrainingRecommendation(

                    capability=capability,

                    frequency=frequency,

                    priority=priority,

                    recommendation=(
                        f"Collect additional data targeting "
                        f"{capability.replace('_',' ')}."
                    ),

                    suggested_training=training,

                    expected_impact=priority

                )

            )


            return EvaluationReport(

            overall_metrics=overall_metrics,

            failure_profile=failure_profile,

            evidence_metrics=evidence_metrics,

            safety_metrics=safety_metrics,

            missing_information_analysis=missing_information,

            positive_findings_analysis=positive_findings,

            failure_signature=signature,

            training_recommendations=recommendations,


training_insights=training_insights

        )

In [ ]:
from collections import Counter
from pydantic import BaseModel
from typing import List


# ============================================================
# Root Cause Insight
# ============================================================

class RootCauseInsight(BaseModel):

    capability: str

    frequency: int

    unsafe_cases: int

    critical_cases: int

    average_claim_support: float

    recommendation: str

    suggested_training: str

    priority: str

In [ ]:
class TrainingInsightEngine:

    def analyze(

        self,

        failure_records: List[FailureRecord]

    ) -> List[RootCauseInsight]:

        capability_stats = {}

        for record in failure_records:

            for capability in record.failure_tags:

                name = capability.value

                if name not in capability_stats:

                    capability_stats[name] = {

                        "count": 0,

                        "unsafe": 0,

                        "critical": 0,

                        "claim_support": []

                    }

                capability_stats[name]["count"] += 1

                if record.unsafe:

                    capability_stats[name]["unsafe"] += 1

                if record.severity == "Critical":

                    capability_stats[name]["critical"] += 1

                capability_stats[name]["claim_support"].append(

                    record.claim_support_rate

                )

                insights = []

                for capability, stats in capability_stats.items():

                    avg_support = sum(

                        stats["claim_support"]

                    ) / len(

                        stats["claim_support"]

                    )

                    # -----------------------------
                    # Priority
                    # -----------------------------

                    if (

                        stats["critical"] >= 20

                        or

                        stats["unsafe"] >= 50

                    ):

                        priority = "HIGH"

                    elif stats["count"] >= 30:

                        priority = "MEDIUM"

                    else:

                        priority = "LOW"

                            # -----------------------------
            # Training Strategy
            # -----------------------------

            if capability in [

                "clinical_reasoning",

                "treatment_planning",

                "diagnosis"

            ]:

                training = "Preference Optimization (DPO)"

                recommendation = (

                    "Generate high-quality preference pairs "

                    "targeting clinical reasoning."

                )

            elif capability in [

                "guideline_adherence",

                "documentation",

                "communication"

            ]:

                training = "Supervised Fine-Tuning (SFT)"

                recommendation = (

                    "Collect supervised demonstrations "

                    "covering guideline-compliant answers."

                )

            elif capability in [

                "hallucination",

                "context_utilization"

            ]:

                training = "RAG / Context Optimization"

                recommendation = (

                    "Improve retrieval quality and "

                    "context grounding."

                )

            else:

                training = "Targeted Data Collection"

                recommendation = (

                    "Collect additional examples "

                    f"covering {capability.replace('_',' ')}."

                )
                insights.append(

                RootCauseInsight(

                    capability=capability,

                    frequency=stats["count"],

                    unsafe_cases=stats["unsafe"],

                    critical_cases=stats["critical"],

                    average_claim_support=round(

                        avg_support,

                        3

                    ),

                    recommendation=recommendation,

                    suggested_training=training,

                    priority=priority

                )

            )

        insights.sort(

            key=lambda x: (

                x.priority,

                x.frequency

            ),

            reverse=True

        )

        return insights